In [1]:
import os
os.chdir("../")
os.getcwd()

'/home/minh_khai/salinity/ai-agent-learning/quick-ai-agent-test'

In [2]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

def load_pdf_files(data):
    loader = DirectoryLoader(
        data, glob="*.pdf", loader_cls=PyPDFLoader
    )
    
    documents = loader.load()
    return documents

extracted_data = load_pdf_files("Medical_AI_Chatbot/data")
len(extracted_data)

/tmp/ipykernel_957/2648548383.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
/home/minh_khai/salinity/ai-agent-learning/quick-ai-agent-test/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


637

In [3]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs : List[Document] = []
    
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content = doc.page_content,
                metadata     = {"source": src}
            )                    
        )
    
    return minimal_docs

minimal_docs = filter_to_minimal_docs(extracted_data)
minimal_docs[0]

Document(metadata={'source': 'Medical_AI_Chatbot/data/Medical_book.pdf'}, page_content='')

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500, chunk_overlap=20
    )
    
    return text_splitter.split_documents(minimal_docs)

texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")
texts_chunk[0]

Number of chunks: 5859


Document(metadata={'source': 'Medical_AI_Chatbot/data/Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION')

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embedding = download_embeddings()
embedding

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1153.14it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [6]:
from dotenv import load_dotenv
load_dotenv()
from langchain_anthropic import ChatAnthropic
from pinecone import Pinecone 

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
PINECONE_API_KEY  = os.getenv("PINECONE_API_KEY")
llm = ChatAnthropic(model="claude-haiku-4-5-20251001")

pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)
pc

In [7]:
from pinecone import ServerlessSpec

index_name = 'medical-ai-chatbot'
if not pc.has_index(name=index_name):
    index = pc.create_index(
        name        = index_name,
        dimension   = 384,
        metric      = 'cosine',
        spec = ServerlessSpec(cloud="aws", region="us-east-1")
    )
    
index = pc.Index(index_name)

In [8]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents = texts_chunk,
    embedding = embedding,
    index_name= index_name
)

In [9]:
from langchain_pinecone import PineconeVectorStore

doc_search = PineconeVectorStore.from_existing_index(
    index_name= index_name,
    embedding = embedding
)

### Adding data to the existing Pinecone index

In [10]:
dswith = Document(
    page_content= "dswithbappy is a youtube channel that provides tutorials on various topics.",
    metadata    = {"source": "Youtube"}
)

docsearch.add_documents(documents=[dswith])
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='e3487728-51b6-4fd6-b308-c062c31345a8', metadata={'source': 'Medical_AI_Chatbot/data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='f80b63b4-a596-40a2-ba0f-0b68e47f67fa', metadata={'source': 'Medical_AI_Chatbot/data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='76ca13ff-25f6-460c-89a3-1ad8827793d1', metadata={'source': 'Medical_AI_Chatbot/data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25')]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

rag_chain = (
    {
        "context": RunnableLambda(lambda x: retriever.invoke(x["input"])), 
        "input": RunnableLambda(lambda x: x["input"])
    }
    | prompt | llm | StrOutputParser()
)

In [19]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response)

# Acromegaly and Gigantism

Acromegaly is a disorder caused by abnormal release of a chemical (growth hormone) from the pituitary gland in the brain, which leads to increased growth in bone and soft tissue along with various other disturbances throughout the body. Gigantism is the related condition that occurs when this excessive growth hormone is present, typically causing abnormal skeletal growth. Both conditions result from pituitary gland dysfunction causing hormonal excess.


In [21]:
response = rag_chain.invoke({"input": "what is Acne?"})
print(response)

Acne is a skin disorder in which the sebaceous glands become inflamed. It is a common condition that can affect various parts of the body, particularly the face. Acne vulgaris is the medical term for this inflammatory skin condition.


In [22]:
response = rag_chain.invoke({"input": "what is the Treatment of Acne?"})
print(response)

# Treatment of Acne

The treatment of acne depends on whether it is mild, moderate, or severe. For **mild noninflammatory acne**, topical medications are used to reduce the formation of new comedones, including tretinoin, benzoyl peroxide, adapalene, or salicylic acid—with tretinoin being especially effective because it increases skin cell turnover. When acne is complicated by inflammation, **topical antibiotics** may be added to the treatment regimen, with improvement typically seen within two to four weeks.


In [23]:
response = rag_chain.invoke({"input": "what is dswithbappy?"})
print(response)

dswithbappy is a YouTube channel that provides tutorials on various topics. Based on the retrieved information, that's all I can tell you about it from the available sources.
